# POST-Filtered Attack Dataset

This notebook extracts the requested classes from `data_capec_multilabel.csv`, keeps only `POST` requests with a usable request body, remaps labels, and saves the merged result as a new CSV file.

In [1]:
import pandas as pd

INPUT_PATH = "data_capec_multilabel.csv"
OUTPUT_PATH = "post_filtered_attack_dataset.csv"

## Load Source Dataset

In [2]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
df.shape

(907815, 38)

## Define Label Mapping

In [3]:
label_mapping = {
    "000 - Normal": 0,
    "66 - SQL Injection": 1,
    "88 - OS Command Injection": 3,
    "242 - Code Injection": 4,
}

label_mapping

{'000 - Normal': 0,
 '66 - SQL Injection': 1,
 '88 - OS Command Injection': 3,
 '242 - Code Injection': 4}

## Build a Reusable Filter Function

In [4]:
def build_subset(dataframe: pd.DataFrame, class_column: str, numeric_label: int) -> pd.DataFrame:
    filtered = dataframe.loc[
        (dataframe[class_column] == 1)
        & (dataframe["request_http_method"] == "POST")
        & dataframe["request_body"].notna(),
        ["request_http_request", "request_body", "request_http_method"],
    ].copy()

    filtered = filtered.rename(
        columns={
            "request_http_request": "url",
            "request_body": "parameter",
            "request_http_method": "request_type",
        }
    )
    filtered["label"] = numeric_label

    return filtered[["url", "parameter", "request_type", "label"]]

## Extract the Requested Classes

In [5]:
subsets = {
    class_column: build_subset(df, class_column, numeric_label)
    for class_column, numeric_label in label_mapping.items()
}

{class_column: subset.shape for class_column, subset in subsets.items()}

{'000 - Normal': (3085, 4),
 '66 - SQL Injection': (9886, 4),
 '88 - OS Command Injection': (4579, 4),
 '242 - Code Injection': (1954, 4)}

## Validate Each Subset

In [ ]:
for class_column, numeric_label in label_mapping.items():
    subset = subsets[class_column]
    assert list(subset.columns) == ["url", "parameter", "request_type", "label"]
    assert subset["request_type"].eq("POST").all()
    assert subset["label"].eq(numeric_label).all()

    source_rows = df.loc[
        (df[class_column] == 1)
        & (df["request_http_method"] == "POST")
        & df["request_body"].notna()
    ]
    assert len(subset) == len(source_rows)

print("Subset validation completed successfully.")

Subset validation completed successfully.


## Merge and Reorder the Final Dataset

In [7]:
final_df = pd.concat(subsets.values(), ignore_index=True)
final_df = final_df[["url", "parameter", "request_type", "label"]]

final_df.shape

(19504, 4)

## Validate Final Output

In [8]:
assert list(final_df.columns) == ["url", "parameter", "request_type", "label"]
assert final_df["request_type"].eq("POST").all()
assert set(final_df["label"].unique()) <= set(label_mapping.values())
assert not final_df.empty

final_df.head()

,url,parameter,request_type,label
0,/blog/index.php/my-account/,username=rafael&password=espa%C3%B1a01&user-re...,POST,0
1,/blog/wp-comments-post.php,comment=&submit=Post+Comment&comment_post_ID=8...,POST,0
2,/blog/wp-comments-post.php,comment=&submit=Post+Comment&comment_post_ID=1...,POST,0
3,/blog/wp-comments-post.php,comment=&submit=Post+Comment&comment_post_ID=8...,POST,0
4,/blog/wp-comments-post.php,comment=&submit=Post+Comment&comment_post_ID=3...,POST,0


## Save the Result to Disk

In [9]:
final_df.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

'post_filtered_attack_dataset.csv'

## Spot Check Sample Rows

In [10]:
for class_column, numeric_label in label_mapping.items():
    print(f"Class: {class_column} -> Label: {numeric_label}")
    display(subsets[class_column].head(3))


Class: 000 - Normal -> Label: 0


,url,parameter,request_type,label
2964,/blog/index.php/my-account/,username=rafael&password=espa%C3%B1a01&user-re...,POST,0
3138,/blog/wp-comments-post.php,comment=&submit=Post+Comment&comment_post_ID=8...,POST,0
3141,/blog/wp-comments-post.php,comment=&submit=Post+Comment&comment_post_ID=1...,POST,0


Class: 66 - SQL Injection -> Label: 1


,url,parameter,request_type,label
19354,/blog/index.php/my-account/edit-password/,username=enragedMandrill8&password=n$Y#pQ5g&us...,POST,1
19357,/blog/index.php/my-account/edit-password/,username=sadTortoise9&password=E4hw4G&user-reg...,POST,1
19825,/c%3A%5C/index.php/my-account/edit-profile/pos...,username=pitifulCoati0&password=Ujr#q4&user-re...,POST,1


Class: 88 - OS Command Injection -> Label: 3


,url,parameter,request_type,label
2961,/cgi-bin/ViewLog.asp,remote_submit_Flag=1&remote_syslog_Flag=1&Rem...,POST,3
4746,/,"<?xml version=""1.0"" encoding=""UTF-8""?>",POST,3
19273,/blog/index.php/my-account/edit-password/,password_current=%2Fetc%2Fpasswd&password_1=ZW...,POST,3


Class: 242 - Code Injection -> Label: 4


,url,parameter,request_type,label
2940,/vendor/phpunit/phpunit/src/Util/PHP/eval-stdi...,"<?=md5(""phpunit"")?>",POST,4
4744,/?-d+allow_url_include%3d1+-d+auto_prepend_fil...,<?php system('echo NmapCVEIdentification');die...,POST,4
103920,/blog/index.php/my-account/edit-password/,password_current=%3C%21--%23EXEC+cmd%3D%22ls+%...,POST,4
